<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #1e3a8a; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Visualización Multivariada: Relaciones entre Múltiples Variables 🕸️
      </h1>
      <p style="margin: 6px 0 0 0; color: #1e3a8a; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Visual Analytics and Critical Thinking
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #1e3a8a; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Módulo 06 🧗
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #2563eb; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Visual%20Analytics%20and%20Critical%20Thinking/06%20-%20Visualizacion%20Avanzada/02_Visualizacion_Multivariada.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## Objetivos de Aprendizaje 🔎

Esta subsección (2.4.3 del syllabus) cubre la **visualización multivariada**: cómo explorar relaciones entre **más de dos variables** al mismo tiempo, cuando un simple scatter plot 2D ya no alcanza.

1. Entender el reto de visualizar más de 2-3 dimensiones en una pantalla 2D.
2. Construir **matrices de dispersión** (*pair plots*) para explorar todas las relaciones bivariadas a la vez.
3. Construir **matrices de correlación** con `heatmap` para cuantificar relaciones lineales.
4. Construir **coordenadas paralelas** (*parallel coordinates*) para comparar observaciones en múltiples ejes.
5. Combinar codificaciones visuales (posición, tamaño, color, forma) en un solo gráfico multivariado.

> 🧗 **Nivel avanzado:** Este módulo asume dominio de los módulos 01-05.

---
## Recursos Recomendados 📚

- [Seaborn — pairplot](https://seaborn.pydata.org/generated/seaborn.pairplot.html)
- [Seaborn — heatmap](https://seaborn.pydata.org/generated/seaborn.heatmap.html)
- [Pandas — parallel_coordinates](https://pandas.pydata.org/docs/reference/api/pandas.plotting.parallel_coordinates.html)
- [Plotly — Parallel Coordinates](https://plotly.com/python/parallel-coordinates-plot/)
- [Power BI — Scatter chart y Decomposition tree](https://learn.microsoft.com/es-es/power-bi/visuals/power-bi-visualization-scatter)
- [Tableau — Multiple Measures](https://help.tableau.com/current/pro/desktop/es-es/multiple_measures.htm)

---
## 1. El Reto de Más de Dos Dimensiones 🧭

Un scatter plot básico codifica 2 variables (eje X, eje Y). Pero muchos problemas reales requieren comparar **4, 5 o más variables simultáneamente** (p.ej. un cliente descrito por edad, ingreso, gasto mensual y antigüedad).

### Estrategias para "aplanar" muchas dimensiones en 2D

| Técnica | Variables que maneja | Cómo codifica cada una |
|---|---|---|
| **Pair plot (matriz de dispersión)** | Todas contra todas | Cada celda = un scatter de 2 variables |
| **Matriz de correlación (heatmap)** | Todas contra todas (resumen numérico) | Color = fuerza y signo de la correlación |
| **Coordenadas paralelas** | Muchas (5-10+) | Un eje vertical por variable, una línea por observación |
| **Codificación combinada** (bubble chart) | Hasta 4-5 en un solo panel | Posición (X,Y) + tamaño + color + forma |

Trabajaremos con un dataset sintético de **clientes** con 5 variables numéricas y 1 categórica.

In [ ]:
# ============================================================
# Dataset sintético multivariado: perfil de clientes
# ============================================================

import numpy as np
import pandas as pd

np.random.seed(123)
n = 200

segmento = np.random.choice(['Premium', 'Estándar', 'Básico'], n, p=[0.2, 0.5, 0.3])
edad = np.random.normal(38, 11, n).clip(18, 75)
ingreso_mensual = np.where(
    segmento == 'Premium', np.random.normal(9_500_000, 1_800_000, n),
    np.where(segmento == 'Estándar', np.random.normal(4_200_000, 900_000, n),
             np.random.normal(1_800_000, 500_000, n))
).clip(800_000, None)
gasto_mensual = ingreso_mensual * np.random.uniform(0.15, 0.45, n)
antiguedad_meses = np.random.gamma(shape=2.0, scale=18, size=n).clip(1, 120)
satisfaccion = (5 + 0.15 * (antiguedad_meses / 12) + np.random.normal(0, 1.1, n)).clip(1, 10)

clientes = pd.DataFrame({
    'segmento': segmento,
    'edad': edad.round(0),
    'ingreso_mensual': ingreso_mensual.round(0),
    'gasto_mensual': gasto_mensual.round(0),
    'antiguedad_meses': antiguedad_meses.round(1),
    'satisfaccion': satisfaccion.round(1),
})

print(clientes.head())
print(f"\n{len(clientes)} clientes | Variables numéricas: edad, ingreso_mensual, gasto_mensual, antiguedad_meses, satisfaccion")
clientes.describe().round(1)

---
## 2. Matriz de Dispersión (*Pair Plot*) 🔲

Un **pair plot** genera una grilla NxN de scatter plots: cada celda (i, j) muestra la relación entre la variable i y la variable j. La diagonal suele mostrar la distribución univariada de cada variable. Es la forma más rápida de detectar correlaciones, clusters y outliers entre muchas variables a la vez.

In [ ]:
# ============================================================
# Pair plot con seaborn — todas las relaciones bivariadas a la vez
# ============================================================

import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_style('whitegrid')

    vars_numericas = ['edad', 'ingreso_mensual', 'gasto_mensual', 'antiguedad_meses', 'satisfaccion']

    g = sns.pairplot(
        clientes, vars=vars_numericas, hue='segmento',
        palette={'Premium': '#1e3a8a', 'Estándar': '#3b82f6', 'Básico': '#94a3b8'},
        diag_kind='kde', height=1.8, plot_kws={'alpha': 0.6, 's': 25}
    )
    g.fig.suptitle('Matriz de Dispersión (Pair Plot): Perfil de Clientes', y=1.02, fontweight='bold')
    plt.show()
except ImportError:
    print("ℹ️  seaborn no está instalado. Instalar con: pip install seaborn")
    print("   Alternativa: pandas.plotting.scatter_matrix(clientes[vars_numericas])")

print("\n💡 Busca en la grilla: relaciones lineales claras (p.ej. ingreso vs. gasto), y si los")
print("   colores (segmentos) forman grupos separados en algún par de variables.")
print("\n📊 En Power BI: no existe un pair plot nativo; se aproxima combinando varios visuales")
print("   de dispersión en una cuadrícula manual, o usando el visual personalizado 'Chart Matrix'.")
print("📈 En Tableau: se construye manualmente como un Dashboard de múltiples hojas de dispersión.")

---
## 3. Matriz de Correlación con Heatmap 🌡️

Cuando el pair plot tiene demasiadas variables para inspeccionar visualmente cada scatter, la **matriz de correlación** resume la fuerza y dirección de la relación lineal entre cada par de variables en un solo número (de -1 a 1), visualizado como un mapa de calor.

In [ ]:
# ============================================================
# Matriz de correlación + heatmap
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

vars_numericas = ['edad', 'ingreso_mensual', 'gasto_mensual', 'antiguedad_meses', 'satisfaccion']
corr = clientes[vars_numericas].corr()
print(corr.round(2))

try:
    import seaborn as sns
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                vmin=-1, vmax=1, square=True, linewidths=0.5,
                cbar_kws={'label': 'Correlación de Pearson'}, ax=ax)
    ax.set_title('Matriz de Correlación: Variables de Clientes', fontweight='bold')
    plt.tight_layout()
    plt.show()
except ImportError:
    # Alternativa sin seaborn: matplotlib puro
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xticks(range(len(vars_numericas))); ax.set_xticklabels(vars_numericas, rotation=45, ha='right')
    ax.set_yticks(range(len(vars_numericas))); ax.set_yticklabels(vars_numericas)
    for i in range(len(vars_numericas)):
        for j in range(len(vars_numericas)):
            ax.text(j, i, f'{corr.values[i, j]:.2f}', ha='center', va='center', fontsize=8)
    plt.colorbar(im, ax=ax, label='Correlación de Pearson')
    ax.set_title('Matriz de Correlación (matplotlib puro)', fontweight='bold')
    plt.tight_layout()
    plt.show()

print("\n📊 En Power BI: visual 'Key Influencers' o matriz manual con medidas DAX de correlación.")
print("📈 En Tableau: Analytics → 'Scatter Plot Matrix' (extensión) o cálculo CORR() manual por par.")

---
## 4. Coordenadas Paralelas (*Parallel Coordinates*) 📶

En un gráfico de **coordenadas paralelas**, cada variable tiene su propio eje vertical (paralelo a los demás), y cada observación es una **línea** que atraviesa todos los ejes. Es especialmente útil para comparar muchas variables (5-10+) sin necesitar una grilla NxN, y para detectar clusters cuando líneas del mismo grupo siguen trayectorias similares.

In [ ]:
# ============================================================
# Coordenadas paralelas con pandas.plotting
# ============================================================

import matplotlib.pyplot as plt
from pandas.plotting import parallel_coordinates

vars_numericas = ['edad', 'ingreso_mensual', 'gasto_mensual', 'antiguedad_meses', 'satisfaccion']

# Normalizamos (0-1) cada variable para que sean comparables en el mismo eje
clientes_norm = clientes.copy()
for v in vars_numericas:
    vmin, vmax = clientes_norm[v].min(), clientes_norm[v].max()
    clientes_norm[v] = (clientes_norm[v] - vmin) / (vmax - vmin)

fig, ax = plt.subplots(figsize=(10, 5))
parallel_coordinates(
    clientes_norm[vars_numericas + ['segmento']], class_column='segmento',
    color=['#1e3a8a', '#3b82f6', '#94a3b8'], alpha=0.4, ax=ax
)
ax.set_title('Coordenadas Paralelas: Perfil de Clientes por Segmento (variables normalizadas 0-1)',
              fontweight='bold')
ax.set_ylabel('Valor normalizado')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

print("💡 Si las líneas de 'Premium' se agrupan en la zona alta de ingreso_mensual y gasto_mensual,")
print("   eso confirma visualmente que el segmento está bien definido por esas variables.")
print("\n📊 En Power BI: no hay coordenadas paralelas nativas; requiere un visual personalizado (AppSource).")
print("📈 En Tableau: se puede construir con un truco de 'Dual Axis' + unión de datos (pivot), o extensión.")

---
## 5. Codificación Visual Combinada: Bubble Charts Multivariados 🔮

Un **bubble chart** bien diseñado codifica hasta 4-5 variables en un solo panel: posición X e Y (2 variables continuas), tamaño de burbuja (una tercera), color (una cuarta, continua o categórica), y opcionalmente forma del marcador (una quinta, categórica).

In [ ]:
# ============================================================
# Bubble chart multivariado: 4 variables en un solo gráfico
# X=ingreso, Y=satisfacción, tamaño=antigüedad, color=segmento
# ============================================================

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 6))
colores_seg = {'Premium': '#1e3a8a', 'Estándar': '#3b82f6', 'Básico': '#94a3b8'}

for seg, color in colores_seg.items():
    sub = clientes[clientes['segmento'] == seg]
    ax.scatter(
        sub['ingreso_mensual'] / 1e6, sub['satisfaccion'],
        s=sub['antiguedad_meses'] * 3, c=color, alpha=0.6,
        edgecolors='white', linewidths=0.6, label=seg
    )

ax.set_xlabel('Ingreso mensual (Millones $)')
ax.set_ylabel('Satisfacción (1-10)')
ax.set_title('Bubble Chart Multivariado: Ingreso × Satisfacción × Antigüedad × Segmento',
              fontweight='bold')
ax.legend(title='Segmento (color)', loc='lower right')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print("💡 4 variables en un gráfico: eje X (ingreso), eje Y (satisfacción),")
print("   tamaño de burbuja (antigüedad), color (segmento).")
print("\n🧩 Nota: esta misma lógica de combinar múltiples codificaciones visuales es la base")
print("   de los paneles de un dashboard multivariado (p.ej. en Dash o Streamlit), donde cada")
print("   gráfico añade una vista adicional sobre el mismo conjunto de observaciones.")
print("\n📊 En Power BI: Visual 'Gráfico de dispersión' — campos Detalles, Tamaño, Leyenda (color).")
print("📈 En Tableau: Marks card — arrastra medidas a Size, Color y Shape simultáneamente.")

---
## 6. Comparativa: Python vs Power BI vs Tableau para Multivariado 🔺

| Capacidad | Python (seaborn/plotly/pandas) | Power BI | Tableau |
|---|---|---|---|
| **Pair plot** | `sns.pairplot()` | No nativo (visual personalizado) | Manual (múltiples hojas) |
| **Matriz de correlación** | `df.corr()` + `sns.heatmap()` | Key Influencers / DAX manual | CORR() por par, manual |
| **Coordenadas paralelas** | `pandas.plotting.parallel_coordinates` | Visual personalizado (AppSource) | Extensión / truco Dual Axis |
| **Codificación combinada (bubble)** | `plt.scatter(s=..., c=...)` | Gráfico de dispersión (Tamaño+Leyenda) | Marks: Size + Color + Shape |
| **Interactividad entre variables** | Plotly (`dimensions=[...]`) | Segmentaciones cruzadas | Acciones de filtro |

> 🚀 Con este cuaderno cerramos la sección 2.4 de Visualización Avanzada: Geoespacial, Temporal y Multivariada.

---
## Resumen y Puntos Clave 🎯

- La visualización multivariada busca representar **más de 2-3 variables** simultáneamente sin perder claridad.
- El **pair plot** muestra todas las relaciones bivariadas a la vez — ideal para exploración inicial.
- La **matriz de correlación** resume la fuerza lineal entre variables en un solo número por par, visualizado con `heatmap`.
- Las **coordenadas paralelas** comparan observaciones completas (todas sus variables) como líneas sobre ejes paralelos.
- Combinar **posición + tamaño + color (+ forma)** en un bubble chart permite codificar 4-5 variables en un solo panel.
- Power BI y Tableau cubren la codificación combinada (bubble charts) de forma nativa, pero pair plots y coordenadas paralelas casi siempre requieren Python o extensiones.

### 🧠 Autoevaluación

1. ¿Qué información revela un pair plot que una sola matriz de correlación no puede mostrar? Da un ejemplo (pista: piensa en relaciones no lineales).
2. Si dos variables tienen una correlación de -0.85, ¿qué significa el signo negativo y qué tan fuerte es esa relación?
3. En un gráfico de coordenadas paralelas, ¿por qué es necesario normalizar las variables (llevarlas a la misma escala, p.ej. 0-1) antes de graficarlas?
4. Diseña mentalmente un bubble chart para un dataset de países: ¿qué variable pondrías en X, en Y, en tamaño y en color? Justifica tu elección.